
# Notebook 04 — V1 Failure Analysis

## 4.1 V1 Data Loading & Inventory

**Research Question:** What evidence is available from the previous attack simulation to explain the reduction from 995 assigned users to 670 simulated users?

**Objectives:**
- Locate the previous simulation datasets, notebooks, and validation reports.
- Identify the approved Stage 2 assignments and final Stage 3 Ground Truth.
- Read the original assignment and quality-assurance manifests.
- Verify the historical population counts before investigating failures.
- Establish a read-only source for subsequent analysis.

**Constraints:**
- Previous simulation files remain unchanged.
- No re-execution of old injection notebooks.
- No Baseline or Future modifications.
- Frequency/event-count analysis only.
- No new scenario assignment or attack injection.

**Expected Output:** A verified V1 file inventory and historical Stage 2–Stage 3 population summary.


In [1]:

# ============================================================
# 4.1 — V1 DATA LOADING & INVENTORY
# Notebook 04: V1 Failure Analysis
# READ-ONLY — No source files will be modified
# ============================================================

from pathlib import Path
import json
import zipfile

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

ROOT = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration"
)

PROJECT_ROOT = ROOT / "New_Attack_Simulation"

REPORT_DIR_04 = (
    PROJECT_ROOT
    / "reports"
    / "04_V1_Failure_Analysis"
)

assert ROOT.is_dir(), f"Root folder not found: {ROOT}"

# ------------------------------------------------------------
# 2. LOCATE THE OLD CLEAR ATTACK SIMULATION
# ------------------------------------------------------------

STAGE2_MANIFEST = (
    "data/stages_output/"
    "stage2_assignment_validation_manifest_r62.json"
)

V1_CANDIDATES = [
    ROOT,
    ROOT / "Clear_Attack_Simulation",
    ROOT / "Clear_Attack_Simulation(3).zip",
    ROOT / "Clear_Attack_Simulation(2).zip",
    ROOT / "Clear_Attack_Simulation.zip",
]

V1_SOURCE = None
V1_SOURCE_TYPE = None

for candidate in V1_CANDIDATES:

    if candidate.is_dir():
        if (candidate / STAGE2_MANIFEST).is_file():
            V1_SOURCE = candidate
            V1_SOURCE_TYPE = "DIRECTORY"
            break

    elif candidate.is_file() and candidate.suffix.lower() == ".zip":
        with zipfile.ZipFile(candidate) as z:
            if any(
                name.endswith(STAGE2_MANIFEST)
                for name in z.namelist()
            ):
                V1_SOURCE = candidate
                V1_SOURCE_TYPE = "ZIP"
                break

assert V1_SOURCE is not None, (
    "Clear_Attack_Simulation not found. "
    f"Checked locations under: {ROOT}"
)

# ------------------------------------------------------------
# 3. BUILD READ-ONLY FILE INDEX
# ------------------------------------------------------------

if V1_SOURCE_TYPE == "DIRECTORY":

    V1_FILE_INDEX = {
        p.relative_to(V1_SOURCE).as_posix(): p
        for p in V1_SOURCE.rglob("*")
        if p.is_file()
    }

else:

    with zipfile.ZipFile(V1_SOURCE) as z:

        zip_members = [
            name for name in z.namelist()
            if not name.endswith("/")
        ]

    manifest_member = next(
        name for name in zip_members
        if name.endswith(STAGE2_MANIFEST)
    )

    zip_prefix = manifest_member[:-len(STAGE2_MANIFEST)]

    V1_FILE_INDEX = {
        name[len(zip_prefix):]: name
        for name in zip_members
        if name.startswith(zip_prefix)
    }

V1_FILES = set(V1_FILE_INDEX)

def read_v1_bytes(relative_path):
    """Read a V1 file without changing the original."""

    if relative_path not in V1_FILE_INDEX:
        raise FileNotFoundError(
            f"Missing V1 artifact: {relative_path}"
        )

    source_item = V1_FILE_INDEX[relative_path]

    if V1_SOURCE_TYPE == "DIRECTORY":
        return source_item.read_bytes()

    with zipfile.ZipFile(V1_SOURCE) as z:
        return z.read(source_item)

# ------------------------------------------------------------
# 4. VERIFY REQUIRED ARTIFACTS
# ------------------------------------------------------------

REQUIRED_FILES = {
    "Scenario catalog":
        "data/stages_output/"
        "approved_scenario_catalog_r62.parquet",

    "Approved scenarios":
        "data/stages_output/"
        "approved_attack_scenarios_r62.csv",

    "Stage 2 assignments":
        "data/stages_output/"
        "validated_scenario_assignments_r62.parquet",

    "Stage 2 manifest":
        STAGE2_MANIFEST,

    "Stage 3 Ground Truth":
        "data/ground_truth/"
        "ground_truth_users.parquet",

    "Scenario design notebook":
        "notebooks/"
        "01_scenario_rules_and_feasibility.ipynb",

    "User assignment notebook":
        "notebooks/"
        "02_user_eligibility_and_assignment.ipynb",

    "Scheduling notebook":
        "notebooks/"
        "03_attack_scheduling_and_injection.ipynb",

    "Validated schedule":
        "validation_inputs/"
        "validated_schedule_339.parquet"
}

file_status = {
    label: path in V1_FILES
    for label, path in REQUIRED_FILES.items()
}

missing_files = [
    label for label, found in file_status.items()
    if not found
]

assert not missing_files, (
    f"Missing required files: {missing_files}"
)

# Find the Stage 3.5 QA summary.

qa_matches = sorted(
    name for name in V1_FILES
    if name.endswith("MUBSIR_Stage3_5_QA_Summary.json")
)

assert len(qa_matches) == 1, (
    f"Expected one QA summary, found {len(qa_matches)}"
)

QA_SUMMARY = qa_matches[0]

# ------------------------------------------------------------
# 5. READ ORIGINAL STAGE COUNTS
# ------------------------------------------------------------

v1_stage2_manifest = json.loads(
    read_v1_bytes(STAGE2_MANIFEST).decode("utf-8-sig")
)

v1_stage3_qa = json.loads(
    read_v1_bytes(QA_SUMMARY).decode("utf-8-sig")
)

v1_assigned_count = int(
    v1_stage2_manifest["validated_assignments"]
)

v1_simulated_count = int(
    v1_stage3_qa["simulated_users"]
)

v1_not_simulated = (
    v1_assigned_count - v1_simulated_count
)

assert v1_assigned_count == 995
assert v1_simulated_count == 670
assert v1_not_simulated == 325

# ------------------------------------------------------------
# 6. FINAL INVENTORY REPORT
# ------------------------------------------------------------

print("=" * 70)
print("NOTEBOOK 04 — V1 DATA LOADING & INVENTORY")
print("=" * 70)

print(f"Source type          : {V1_SOURCE_TYPE}")
print(f"Source path          : {V1_SOURCE}")
print(f"Files discovered     : {len(V1_FILES):,}")
print(f"Required artifacts   : {sum(file_status.values())}/{len(REQUIRED_FILES)}")

print("\nHISTORICAL POPULATION")
print("-" * 70)

print(f"Stage 2 assignments  : {v1_assigned_count:,}")
print(f"Stage 3 simulated    : {v1_simulated_count:,}")
print(f"Not simulated        : {v1_not_simulated:,}")

print("\nHISTORICAL VALIDATION")
print("-" * 70)

print(
    "Stage 2 status      :",
    v1_stage2_manifest.get("status")
)

print(
    "Stage 3.5 QA status :",
    v1_stage3_qa.get("overall_status")
)

print("\nSAFETY & INTEGRITY")
print("-" * 70)

print("Required files       : PASS")
print("Historical counts    : PASS")
print("Old notebooks run    : NO")
print("Source modifications : NONE")
print("New files written    : NONE")

print("\nSECTION 4.1 STATUS: PASS")


NOTEBOOK 04 — V1 DATA LOADING & INVENTORY
Source type          : DIRECTORY
Source path          : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Clear_Attack_Simulation
Files discovered     : 28
Required artifacts   : 9/9

HISTORICAL POPULATION
----------------------------------------------------------------------
Stage 2 assignments  : 995
Stage 3 simulated    : 670
Not simulated        : 325

HISTORICAL VALIDATION
----------------------------------------------------------------------
Stage 2 status      : APPROVED_WITH_TARGET_SHORTFALL
Stage 3.5 QA status : PASS

SAFETY & INTEGRITY
----------------------------------------------------------------------
Required files       : PASS
Historical counts    : PASS
Old notebooks run    : NO
Source modifications : NONE
New files written    : NONE

SECTION 4.1 STATUS: PASS



## 4.2 V1 Population Reconciliation

**Research Question:** How do the previous simulation's assigned and simulated users overlap with the 995 current multi-channel candidates?

**Objectives:**
- Validate unique user identities in V1 Stage 2 and Stage 3.
- Identify assigned users who did not reach simulation.
- Compare the V1 population with the current 995 candidates from Notebook 03.
- Quantify overlaps and differences without assuming that equal population counts mean identical users.

**Inputs:**
- V1 validated scenario assignments.
- V1 final user-level Ground Truth.
- Notebook 03 candidate behavior profile.

**Constraints:**
- Read-only analysis.
- No modifications to V1, Baseline, Future, or existing reports.
- No scenario reassignment or attack injection.

**Expected Output:** Validated user-set reconciliation and population overlap counts.


In [2]:

# ============================================================
# 4.2 — V1 POPULATION RECONCILIATION
# ============================================================

import pandas as pd
from io import BytesIO

print("=" * 72)
print("SECTION 4.2 — V1 POPULATION RECONCILIATION")
print("=" * 72)

# ------------------------------------------------------------
# 1. LOAD USER-LEVEL SOURCES (READ-ONLY)
# ------------------------------------------------------------

v1_assignments = pd.read_parquet(
    BytesIO(
        read_v1_bytes(
            "data/stages_output/"
            "validated_scenario_assignments_r62.parquet"
        )
    )
)

v1_ground_truth_users = pd.read_parquet(
    BytesIO(
        read_v1_bytes(
            "data/ground_truth/ground_truth_users.parquet"
        )
    )
)

current_candidates_path = (
    PROJECT_ROOT
    / "reports"
    / "03_Channel_Behavior_Audit"
    / "candidate_behavior_profile.parquet"
)

assert current_candidates_path.is_file(), (
    f"Current candidate profile not found: "
    f"{current_candidates_path}"
)

current_candidates = pd.read_parquet(
    current_candidates_path
)

# ------------------------------------------------------------
# 2. VERIFY USER IDENTIFIERS
# ------------------------------------------------------------

sources = {
    "V1 Stage 2": v1_assignments,
    "V1 Stage 3": v1_ground_truth_users,
    "Current candidates": current_candidates
}

user_sets = {}

for source_name, df in sources.items():

    assert "user" in df.columns, (
        f"{source_name}: missing 'user' column. "
        f"Available columns: {list(df.columns)}"
    )

    assert df["user"].notna().all(), (
        f"{source_name}: null user identifiers found."
    )

    normalized_ids = (
        df["user"]
        .astype("string")
        .str.strip()
    )

    assert normalized_ids.ne("").all(), (
        f"{source_name}: empty user identifiers found."
    )

    assert normalized_ids.is_unique, (
        f"{source_name}: duplicate user identifiers found."
    )

    user_sets[source_name] = set(
        normalized_ids.astype(str)
    )

v1_assigned_users = user_sets["V1 Stage 2"]
v1_simulated_users = user_sets["V1 Stage 3"]
current_candidate_users = user_sets["Current candidates"]

# ------------------------------------------------------------
# 3. VERIFY HISTORICAL COUNTS
# ------------------------------------------------------------

assert len(v1_assigned_users) == 995
assert len(v1_simulated_users) == 670
assert len(current_candidate_users) == 995

# Every simulated user must belong to V1 assignments.
assert v1_simulated_users <= v1_assigned_users, (
    "V1 Stage 3 contains users absent from Stage 2."
)

v1_not_simulated_users = (
    v1_assigned_users - v1_simulated_users
)

assert len(v1_not_simulated_users) == 325

# ------------------------------------------------------------
# 4. COMPARE V1 WITH CURRENT CANDIDATES
# ------------------------------------------------------------

assigned_in_current = (
    v1_assigned_users & current_candidate_users
)

simulated_in_current = (
    v1_simulated_users & current_candidate_users
)

not_simulated_in_current = (
    v1_not_simulated_users & current_candidate_users
)

current_not_in_v1 = (
    current_candidate_users - v1_assigned_users
)

v1_not_in_current = (
    v1_assigned_users - current_candidate_users
)

# Mutually exclusive groups within the CURRENT 995.
current_groups = {
    "Previously simulated":
        simulated_in_current,

    "Assigned in V1, not simulated":
        not_simulated_in_current,

    "Not assigned in V1":
        current_not_in_v1
}

assert (
    sum(len(group) for group in current_groups.values())
    == 995
), "Current candidate groups do not reconcile."

assert not (
    current_groups["Previously simulated"]
    & current_groups["Assigned in V1, not simulated"]
)

assert not (
    current_groups["Previously simulated"]
    & current_groups["Not assigned in V1"]
)

assert not (
    current_groups["Assigned in V1, not simulated"]
    & current_groups["Not assigned in V1"]
)

# ------------------------------------------------------------
# 5. CONCISE RESULTS
# ------------------------------------------------------------

print("\nINPUT POPULATIONS")
print("-" * 72)

print(f"V1 Stage 2 assigned users   : {len(v1_assigned_users):,}")
print(f"V1 Stage 3 simulated users : {len(v1_simulated_users):,}")
print(f"V1 not simulated users     : {len(v1_not_simulated_users):,}")
print(f"Current multi-channel users: {len(current_candidate_users):,}")

print("\nV1 VS CURRENT POPULATION")
print("-" * 72)

print(f"V1 assigned in current     : {len(assigned_in_current):,}")
print(f"V1 simulated in current    : {len(simulated_in_current):,}")
print(f"V1 unsimulated in current  : {len(not_simulated_in_current):,}")
print(f"Current users absent in V1 : {len(current_not_in_v1):,}")
print(f"V1 assigned absent current : {len(v1_not_in_current):,}")

print("\nCURRENT 995 — EXCLUSIVE GROUPS")
print("-" * 72)

for group_name, users in current_groups.items():
    print(f"{group_name:<34}: {len(users):,}")

print("\nVALIDATION")
print("-" * 72)

print("Unique identifiers          : PASS")
print("Historical counts           : PASS")
print("Stage 3 subset of Stage 2   : PASS")
print("Current population partition: PASS")
print("Source files modified       : NONE")
print("Files written               : NONE")

print("\nSECTION 4.2 STATUS: PASS")


SECTION 4.2 — V1 POPULATION RECONCILIATION

INPUT POPULATIONS
------------------------------------------------------------------------
V1 Stage 2 assigned users   : 995
V1 Stage 3 simulated users : 670
V1 not simulated users     : 325
Current multi-channel users: 995

V1 VS CURRENT POPULATION
------------------------------------------------------------------------
V1 assigned in current     : 995
V1 simulated in current    : 670
V1 unsimulated in current  : 325
Current users absent in V1 : 0
V1 assigned absent current : 0

CURRENT 995 — EXCLUSIVE GROUPS
------------------------------------------------------------------------
Previously simulated              : 670
Assigned in V1, not simulated     : 325
Not assigned in V1                : 0

VALIDATION
------------------------------------------------------------------------
Unique identifiers          : PASS
Historical counts           : PASS
Stage 3 subset of Stage 2   : PASS
Current population partition: PASS
Source files modified   


## 4.3 V1 Rules Reconstruction

**Research Question:** Which predefined scenario and scheduling constraints were imposed in V1 before checking users' actual daily channel availability?

**Objectives:**
- Recover the original scenario rules directly from V1 notebooks.
- Verify durations, attack-off days, switching targets, and same-day multichannel requirements.
- Confirm that the approved scenario catalog follows these rules.
- Examine how the 995 assigned users were distributed across the predefined configurations.
- Identify constraints that require failure-impact analysis in the following sections.

**Inputs:**
- V1 scenario design notebook.
- V1 user assignment and scheduling notebooks.
- Approved scenario catalog.
- Validated Stage 2 assignments.

**Constraints:**
- Read-only inspection of old notebook source code.
- No execution of V1 notebooks.
- No modification of existing data.
- Frequency/event-count perspective only.

**Expected Output:** A validated V1 rule inventory and its distribution among the 995 assigned users.


In [3]:

# ============================================================
# 4.3 — V1 RULES RECONSTRUCTION
# ============================================================

import ast
import json
import pandas as pd
from io import BytesIO

print("=" * 72)
print("SECTION 4.3 — V1 RULES RECONSTRUCTION")
print("=" * 72)

# ------------------------------------------------------------
# 1. LOAD ORIGINAL NOTEBOOK SOURCE (READ-ONLY)
# ------------------------------------------------------------

V1_NOTEBOOK_PATHS = {
    "scenario_design":
        "notebooks/01_scenario_rules_and_feasibility.ipynb",

    "assignment":
        "notebooks/02_user_eligibility_and_assignment.ipynb",

    "scheduling":
        "notebooks/03_attack_scheduling_and_injection.ipynb"
}

v1_notebooks_43 = {
    name: json.loads(
        read_v1_bytes(path).decode("utf-8-sig")
    )
    for name, path in V1_NOTEBOOK_PATHS.items()
}

# Extract a literal configuration from old source code
# WITHOUT executing any old notebook cell.

def extract_literal_rule_43(notebook, variable_name):

    matches = []

    for cell_index, cell in enumerate(notebook["cells"]):

        if cell["cell_type"] != "code":
            continue

        source = "".join(cell["source"])

        try:
            tree = ast.parse(source)
        except SyntaxError:
            continue

        for node in tree.body:

            if not isinstance(node, ast.Assign):
                continue

            target_names = [
                target.id
                for target in node.targets
                if isinstance(target, ast.Name)
            ]

            if variable_name not in target_names:
                continue

            try:
                value = ast.literal_eval(node.value)
            except (ValueError, TypeError):
                continue

            matches.append((cell_index, value))

    assert len(matches) == 1, (
        f"Expected one definition of {variable_name}, "
        f"found {len(matches)}."
    )

    return matches[0][1]


design_nb_43 = v1_notebooks_43["scenario_design"]

CHANNELS_43 = extract_literal_rule_43(
    design_nb_43, "CHANNELS"
)

DURATIONS_43 = extract_literal_rule_43(
    design_nb_43, "ATTACK_DURATIONS"
)

WORKDAY_RULES_43 = extract_literal_rule_43(
    design_nb_43, "WORKDAY_CONDITIONS"
)

OFF_RULES_43 = extract_literal_rule_43(
    design_nb_43, "OFF_DAY_RULES"
)

SWITCH_RANGES_43 = extract_literal_rule_43(
    design_nb_43, "CANDIDATE_RANGES"
)["C"]

FIVE_DAY_EXCEPTION_43 = extract_literal_rule_43(
    design_nb_43, "FIVE_DAY_EXCEPTION"
)

MULTI_RULES_43 = extract_literal_rule_43(
    design_nb_43, "MULTICHANNEL_RULES"
)

STEALTH_RULES_43 = extract_literal_rule_43(
    design_nb_43, "STEALTH_MULTIPLIERS"
)

# ------------------------------------------------------------
# 2. LOAD THE ORIGINAL APPROVED SCENARIO CATALOG
# ------------------------------------------------------------

v1_catalog_43 = pd.read_parquet(
    BytesIO(
        read_v1_bytes(
            "data/stages_output/"
            "approved_scenario_catalog_r62.parquet"
        )
    )
)

v1_assigned_43 = v1_assignments.copy()

required_catalog_columns_43 = {
    "scenario_id",
    "duration",
    "off_days",
    "active_days",
    "single_channel_days",
    "two_channel_days",
    "three_channel_days",
    "switching_level",
    "target_switches",
    "stealth_level",
    "stealth_multiplier"
}

assert required_catalog_columns_43.issubset(
    v1_catalog_43.columns
)

assert v1_catalog_43["scenario_id"].is_unique
assert len(v1_catalog_43) == 1143
assert len(v1_assigned_43) == 995
assert v1_assigned_43["user"].is_unique

# ------------------------------------------------------------
# 3. VERIFY THE CATALOG AGAINST ORIGINAL RULES
# ------------------------------------------------------------

rule_checks_43 = {
    "duration": [],
    "off_days": [],
    "day_partition": [],
    "multichannel": [],
    "switching": [],
    "stealth": []
}

for row in v1_catalog_43.itertuples(index=False):

    duration = int(row.duration)
    off_days = int(row.off_days)
    active_days = int(row.active_days)

    two_days = int(row.two_channel_days)
    three_days = int(row.three_channel_days)
    single_days = int(row.single_channel_days)

    switches = int(row.target_switches)
    level = str(row.switching_level)

    rule_checks_43["duration"].append(
        duration in DURATIONS_43
    )

    rule_checks_43["off_days"].append(
        off_days in OFF_RULES_43[duration]
    )

    rule_checks_43["day_partition"].append(
        active_days + off_days == duration
        and single_days + two_days + three_days
        == active_days
    )

    rule_checks_43["multichannel"].append(
        (two_days, three_days)
        in MULTI_RULES_43[duration]
    )

    if duration == 5:

        valid_switching = (
            switches == FIVE_DAY_EXCEPTION_43[level]
        )

    else:

        rate = switches / (active_days - 1)
        minimum, maximum = SWITCH_RANGES_43[level]

        valid_switching = (
            minimum - 1e-9
            <= rate
            <= maximum + 1e-9
        )

    rule_checks_43["switching"].append(
        valid_switching
    )

    rule_checks_43["stealth"].append(
        abs(
            float(row.stealth_multiplier)
            - STEALTH_RULES_43[str(row.stealth_level)]
        ) < 1e-9
    )

for check_name, results in rule_checks_43.items():

    assert all(results), (
        f"Catalog rule mismatch: {check_name} "
        f"({sum(not item for item in results)} failures)"
    )

# ------------------------------------------------------------
# 4. VERIFY HOW STAGE 3 ENFORCED THE RULES
# ------------------------------------------------------------

scheduling_source_43 = "\n".join(
    "".join(cell["source"])
    for cell in v1_notebooks_43["scheduling"]["cells"]
    if cell["cell_type"] == "code"
)

stage3_evidence_43 = {
    "Aggregate multichannel capacity":
        "available_multichannel_days"
        in scheduling_source_43,

    "Aggregate three-channel capacity":
        "three_channel_capacity"
        in scheduling_source_43,

    "Window-level active-day requirements":
        "active >= required_active"
        in scheduling_source_43,

    "Window-level multichannel requirements":
        "multi >= required_multi"
        in scheduling_source_43,

    "Window-level three-channel requirements":
        "three >= required_three"
        in scheduling_source_43,

    "Exact switching feasibility":
        "reachable_switch_counts_333"
        in scheduling_source_43,

    "Exact scheduling solver":
        "solve_window_337"
        in scheduling_source_43
}

assert all(stage3_evidence_43.values()), (
    "Some expected Stage 3 rule checks were not found."
)

# ------------------------------------------------------------
# 5. JOIN ASSIGNMENTS WITH THEIR ACTUAL SCENARIO RULES
# ------------------------------------------------------------

v1_assigned_rules_43 = v1_assigned_43.merge(
    v1_catalog_43[
        list(required_catalog_columns_43)
    ],
    on="scenario_id",
    how="left",
    validate="many_to_one",
    suffixes=("_assignment", "")
)

assert len(v1_assigned_rules_43) == 995

assert v1_assigned_rules_43[
    "three_channel_days"
].notna().all()

v1_assigned_rules_43["requires_same_day_three"] = (
    v1_assigned_rules_43["three_channel_days"].gt(0)
)

v1_assigned_rules_43["long_duration"] = (
    v1_assigned_rules_43["duration"].ge(40)
)

# ------------------------------------------------------------
# 6. RULE SUMMARY BY DURATION
# ------------------------------------------------------------

summary_rows_43 = []

for duration in DURATIONS_43:

    catalog_subset = v1_catalog_43.loc[
        v1_catalog_43["duration"] == duration
    ]

    assigned_subset = v1_assigned_rules_43.loc[
        v1_assigned_rules_43["duration"] == duration
    ]

    multi_configurations = [
        f"{two}x2ch+{three}x3ch"
        for two, three in MULTI_RULES_43[duration]
    ]

    summary_rows_43.append({
        "Duration": duration,
        "Off-days": (
            f"{min(OFF_RULES_43[duration])}-"
            f"{max(OFF_RULES_43[duration])}"
        ),
        "Same-day requirements":
            " / ".join(multi_configurations),
        "Catalog scenarios": len(catalog_subset),
        "Assigned users": len(assigned_subset)
    })

v1_rules_summary_43 = pd.DataFrame(
    summary_rows_43
)

assert (
    v1_rules_summary_43["Catalog scenarios"].sum()
    == 1143
)

assert (
    v1_rules_summary_43["Assigned users"].sum()
    == 995
)

# ------------------------------------------------------------
# 7. CONCISE OUTPUT
# ------------------------------------------------------------

print("\nA. ORIGINAL SCENARIO CONFIGURATION")
print("-" * 72)

print("Channels           :", CHANNELS_43)
print("Durations          :", DURATIONS_43)
print("Workday conditions :", WORKDAY_RULES_43)

print("\nApproved switching ranges (Range C):")

for level, bounds in SWITCH_RANGES_43.items():
    print(
        f"  {level:<8}: "
        f"{bounds[0]:.0%}–{bounds[1]:.0%}"
    )

print("Five-day exception :", FIVE_DAY_EXCEPTION_43)
print("Stealth multipliers:", STEALTH_RULES_43)

print("\nB. RULES BY DURATION")
print("-" * 72)

print(
    v1_rules_summary_43.to_string(index=False)
)

print("\nC. ASSIGNED USER RULE EXPOSURE")
print("-" * 72)

print(
    "Assigned users                  :",
    len(v1_assigned_rules_43)
)

print(
    "Long-duration assignments       :",
    int(v1_assigned_rules_43["long_duration"].sum())
)

print(
    "Required same-day three-channel :",
    int(
        v1_assigned_rules_43[
            "requires_same_day_three"
        ].sum()
    )
)

print("\nAssignment groups:")

print(
    v1_assigned_rules_43[
        "assignment_group"
    ].value_counts().to_string()
)

print("\nD. STAGE 3 RULE ENFORCEMENT")
print("-" * 72)

for name, found in stage3_evidence_43.items():
    print(f"{name}: {'FOUND' if found else 'MISSING'}")

print("\nVALIDATION")
print("-" * 72)

print("Source configurations extracted : PASS")
print("Catalog rule consistency        : PASS")
print("All 1,143 scenarios verified    : PASS")
print("All 995 assignments reconciled  : PASS")
print("Stage 3 source checks located   : PASS")
print("Old notebooks executed          : NO")
print("Source files modified           : NONE")
print("Files written                   : NONE")

print("\nSECTION 4.3 STATUS: PASS")


SECTION 4.3 — V1 RULES RECONSTRUCTION

A. ORIGINAL SCENARIO CONFIGURATION
------------------------------------------------------------------------
Channels           : ('Email', 'USB', 'HTTP')
Durations          : (5, 10, 20, 30, 40, 50)
Workday conditions : {'is_weekend': 0, 'logon_count': '> 0'}

Approved switching ranges (Range C):
  Low     : 10%–25%
  Medium  : 30%–45%
  High    : 50%–65%
Five-day exception : {'Low': 1, 'Medium': 2, 'High': 3}
Stealth multipliers: {'High': 0.5, 'Medium': 1.0, 'Low': 1.5}

B. RULES BY DURATION
------------------------------------------------------------------------
 Duration Off-days                                 Same-day requirements  Catalog scenarios  Assigned users
        5      0-0                                           0x2ch+0x3ch                  9             176
       10      1-2                                           1x2ch+0x3ch                 24             176
       20      2-4                                           2x2ch


## 4.4 V1 User Exclusion Analysis

**Research Question:** Where did the 325 users fail to reach simulation, and how did non-simulation outcomes vary across the original scenario configurations?

**Objectives:**
- Identify the 325 assigned users who were not simulated.
- Reconstruct the Stage 3 population losses from saved notebook execution reports.
- Compare simulated and non-simulated users by attack duration and same-day channel requirements.
- Distinguish documented stage-level losses from individual failure causes not preserved in exported files.
- Prepare a user-level outcome profile for subsequent failure analysis.

**Inputs:**
- V1 Stage 2 assignments and scenario rules.
- V1 Stage 3 Ground Truth user identities.
- Saved execution reports in the original scheduling notebook.

**Constraints:**
- Read-only.
- No old notebook execution or attack injection.
- No modification to Baseline, Future, or previous outputs.
- Do not infer individual failure causes from aggregate stage counts.

**Expected Output:** A validated 995-user outcome profile, duration-level success rates, and documented Stage 3 exclusion breakdown.


In [4]:

# ============================================================
# 4.4 — V1 USER EXCLUSION ANALYSIS
# ============================================================

import re
import pandas as pd

print("=" * 72)
print("SECTION 4.4 — V1 USER EXCLUSION ANALYSIS")
print("=" * 72)

# ------------------------------------------------------------
# 1. BUILD USER-LEVEL OUTCOME PROFILE
# ------------------------------------------------------------

profile_columns_44 = [
    "user",
    "scenario_id",
    "duration",
    "active_days",
    "off_days",
    "two_channel_days",
    "three_channel_days",
    "target_switches",
    "switching_level",
    "stealth_level",
    "assignment_group"
]

assert set(profile_columns_44).issubset(
    v1_assigned_rules_43.columns
), "Required original assignment columns are missing."

v1_user_outcomes_44 = (
    v1_assigned_rules_43[profile_columns_44]
    .copy()
)

v1_user_outcomes_44["user"] = (
    v1_user_outcomes_44["user"]
    .astype("string")
    .str.strip()
)

v1_user_outcomes_44["v1_result"] = (
    v1_user_outcomes_44["user"]
    .map(
        lambda u: (
            "Simulated"
            if u in v1_simulated_users
            else "Not simulated"
        )
    )
)

v1_user_outcomes_44["requires_same_day_3ch"] = (
    v1_user_outcomes_44["three_channel_days"] > 0
)

assert len(v1_user_outcomes_44) == 995
assert v1_user_outcomes_44["user"].is_unique
assert v1_user_outcomes_44["v1_result"].eq("Simulated").sum() == 670
assert v1_user_outcomes_44["v1_result"].eq("Not simulated").sum() == 325

v1_unreached_44 = v1_user_outcomes_44.loc[
    v1_user_outcomes_44["v1_result"] == "Not simulated"
].copy()

assert set(v1_unreached_44["user"]) == v1_not_simulated_users

# ------------------------------------------------------------
# 2. READ SAVED STAGE 3 EXECUTION REPORTS
#    This reads notebook outputs, not notebook code execution.
# ------------------------------------------------------------

old_scheduling_nb_44 = v1_notebooks_43["scheduling"]

def saved_step_output_44(step_marker):

    matches = []

    for cell in old_scheduling_nb_44["cells"]:

        if cell.get("cell_type") != "code":
            continue

        stream_parts = []

        for output in cell.get("outputs", []):

            if output.get("output_type") != "stream":
                continue

            value = output.get("text", "")
            stream_parts.append(
                "".join(value)
                if isinstance(value, list)
                else str(value)
            )

        report = "\n".join(stream_parts)

        if step_marker in report:
            matches.append(report)

    assert len(matches) == 1, (
        f"Expected one saved report for {step_marker}; "
        f"found {len(matches)}."
    )

    return matches[0]


def report_number_44(report, pattern, label):

    matches = re.findall(pattern, report)

    assert len(matches) == 1, (
        f"Cannot uniquely recover '{label}' "
        "from the original saved execution report."
    )

    return int(matches[0])


stage31_report_44 = saved_step_output_44(
    "STAGE 3 — STEP 3.1.7:"
)

stage32_report_44 = saved_step_output_44(
    "STAGE 3 — STEP 3.2.4:"
)

stage33_report_44 = saved_step_output_44(
    "STAGE 3 — STEP 3.3.9:"
)

# Recover historical population counts.

stage31_users_44 = report_number_44(
    stage31_report_44,
    r"Unified preliminary-pass users:\s*(\d+)",
    "Stage 3.1 users"
)

stage32_users_44 = report_number_44(
    stage32_report_44,
    r"Updated provisional window-pass cohort:\s*(\d+)",
    "Stage 3.2 users"
)

stage33_users_44 = report_number_44(
    stage33_report_44,
    r"Validated schedules:\s*(\d+)",
    "Stage 3.3 users"
)

exact_infeasible_44 = report_number_44(
    stage33_report_44,
    r"INFEASIBLE[^\n]*?:\s*(\d+)",
    "Exact scheduling infeasible"
)

exact_incomplete_44 = report_number_44(
    stage33_report_44,
    r"INCOMPLETE[^\n]*?:\s*(\d+)",
    "Exact scheduling incomplete"
)

previously_deferred_44 = report_number_44(
    stage33_report_44,
    r"Previously deferred:\s*(\d+)",
    "Previously deferred users"
)

# ------------------------------------------------------------
# 3. RECONCILE EXCLUSIVE STAGE LOSSES
# ------------------------------------------------------------

stage31_loss_44 = 995 - stage31_users_44
stage32_loss_44 = stage31_users_44 - stage32_users_44
stage33_loss_44 = stage32_users_44 - stage33_users_44

assert (
    stage31_users_44,
    stage32_users_44,
    stage33_users_44
) == (925, 749, 670)

assert stage31_loss_44 + stage32_loss_44 == previously_deferred_44

assert stage33_loss_44 == (
    exact_infeasible_44 + exact_incomplete_44
)

assert (
    stage31_loss_44
    + stage32_loss_44
    + stage33_loss_44
) == 325

v1_stage_losses_44 = pd.DataFrame([
    {
        "Stage": "3.1 — Preliminary feasibility",
        "Users_remaining": stage31_users_44,
        "Users_lost": stage31_loss_44
    },
    {
        "Stage": "3.2 — Window feasibility",
        "Users_remaining": stage32_users_44,
        "Users_lost": stage32_loss_44
    },
    {
        "Stage": "3.3 — Exact scheduling",
        "Users_remaining": stage33_users_44,
        "Users_lost": stage33_loss_44
    }
])

# ------------------------------------------------------------
# 4. ANALYZE OUTCOMES BY ORIGINAL DURATION
# ------------------------------------------------------------

v1_duration_outcomes_44 = pd.crosstab(
    v1_user_outcomes_44["duration"],
    v1_user_outcomes_44["v1_result"]
).reindex(
    columns=["Simulated", "Not simulated"],
    fill_value=0
)

v1_duration_outcomes_44["Total"] = (
    v1_duration_outcomes_44.sum(axis=1)
)

v1_duration_outcomes_44["Success_pct"] = (
    100
    * v1_duration_outcomes_44["Simulated"]
    / v1_duration_outcomes_44["Total"]
).round(1)

# Validate assignment totals against the original manifest.

for duration_str, count in v1_stage2_manifest[
    "duration_distribution"
].items():

    assert (
        v1_duration_outcomes_44.loc[
            int(duration_str), "Total"
        ] == count
    )

assert v1_duration_outcomes_44["Simulated"].sum() == 670
assert v1_duration_outcomes_44["Not simulated"].sum() == 325

# ------------------------------------------------------------
# 5. ANALYZE SAME-DAY THREE-CHANNEL REQUIREMENTS
# ------------------------------------------------------------

v1_channel_requirement_outcomes_44 = pd.crosstab(
    v1_user_outcomes_44["requires_same_day_3ch"],
    v1_user_outcomes_44["v1_result"]
).reindex(
    columns=["Simulated", "Not simulated"],
    fill_value=0
)

v1_channel_requirement_outcomes_44["Total"] = (
    v1_channel_requirement_outcomes_44.sum(axis=1)
)

v1_channel_requirement_outcomes_44["Success_pct"] = (
    100
    * v1_channel_requirement_outcomes_44["Simulated"]
    / v1_channel_requirement_outcomes_44["Total"]
).round(1)

assert (
    v1_channel_requirement_outcomes_44["Total"].sum()
    == 995
)

# ------------------------------------------------------------
# 6. PRINT CONCISE RESULTS
# ------------------------------------------------------------

print("\nA. V1 USER OUTCOMES")
print("-" * 72)

print(f"Assigned users       : {len(v1_user_outcomes_44):,}")
print(f"Simulated users      : {len(v1_simulated_users):,}")
print(f"Not simulated users  : {len(v1_unreached_44):,}")

print("\nB. DOCUMENTED STAGE LOSSES")
print("-" * 72)

print(v1_stage_losses_44.to_string(index=False))

print("\nFinal scheduling breakdown:")
print(f"  INFEASIBLE under V1 rules : {exact_infeasible_44}")
print(f"  INCOMPLETE                : {exact_incomplete_44}")
print(f"  Previously deferred       : {previously_deferred_44}")

print("\nC. OUTCOMES BY ORIGINAL DURATION")
print("-" * 72)

print(v1_duration_outcomes_44.to_string())

print("\nD. SAME-DAY THREE-CHANNEL REQUIREMENT")
print("-" * 72)

readable_channel_table_44 = (
    v1_channel_requirement_outcomes_44.copy()
)

readable_channel_table_44.index = (
    readable_channel_table_44.index.map({
        False: "Not required",
        True: "Required"
    })
)

print(readable_channel_table_44.to_string())

print("\nE. EVIDENCE LIMITATION")
print("-" * 72)

print(
    "All 325 non-simulated users are identified."
)
print(
    "Stage losses are recovered from saved V1 reports."
)
print(
    "Individual stage-specific failure causes are not "
    "assigned without original per-user evidence."
)

print("\nVALIDATION")
print("-" * 72)

print("User-level outcomes       : PASS")
print("Stage population balance  : PASS")
print("325-user loss balance     : PASS")
print("Duration reconciliation   : PASS")
print("Channel-group totals      : PASS")
print("Old notebooks executed    : NO")
print("Files written             : NONE")
print("Source modifications      : NONE")

print("\nSECTION 4.4 STATUS: PASS")


SECTION 4.4 — V1 USER EXCLUSION ANALYSIS

A. V1 USER OUTCOMES
------------------------------------------------------------------------
Assigned users       : 995
Simulated users      : 670
Not simulated users  : 325

B. DOCUMENTED STAGE LOSSES
------------------------------------------------------------------------
                        Stage  Users_remaining  Users_lost
3.1 — Preliminary feasibility              925          70
     3.2 — Window feasibility              749         176
       3.3 — Exact scheduling              670          79

Final scheduling breakdown:
  INFEASIBLE under V1 rules : 78
  INCOMPLETE                : 1
  Previously deferred       : 246

C. OUTCOMES BY ORIGINAL DURATION
------------------------------------------------------------------------
v1_result  Simulated  Not simulated  Total  Success_pct
duration                                               
5                150             26    176         85.2
10               131             45    176  


## 4.5 Scenario Feasibility Failures

**Research Question:** Which preliminary activity and temporal-window requirements prevented V1 from progressing to exact attack scheduling?

**Objectives:**
- Recover preliminary feasibility checks and their overlapping failures.
- Examine failures by assigned scenario duration.
- Identify why initial attack-window screening rejected users.
- Quantify users recovered through alternative scenario selection.
- Reconcile the 246 users deferred before exact switching and scheduling.

**Evidence Rules:**
- Use saved V1 execution reports only.
- Distinguish initial failures from final unresolved cases.
- Do not assign aggregate failure reasons to individual users.
- Do not rerun feasibility calculations already covered by Notebook 02.

**Expected Output:** Validated preliminary and window-failure diagnostics, including recovery results.


In [5]:

# ============================================================
# 4.5 — SCENARIO FEASIBILITY FAILURES
# Read-only reconstruction from saved V1 notebook outputs
# ============================================================

import re
import pandas as pd

print("=" * 72)
print("SECTION 4.5 — SCENARIO FEASIBILITY FAILURES")
print("=" * 72)

# ------------------------------------------------------------
# 1. READ SAVED V1 REPORTS — NO OLD CODE EXECUTION
# ------------------------------------------------------------

reports_45 = {
    step: saved_step_output_44(
        f"STAGE 3 — STEP {step}:"
    )
    for step in [
        "3.1.5", "3.1.6", "3.1.7",
        "3.2.1", "3.2.2", "3.2.3", "3.2.4"
    ]
}

def number_45(step, pattern, label):
    return report_number_44(
        reports_45[step], pattern, label
    )

# Locate a saved displayed table inside a V1 code cell.
def saved_table_45(step, required_headers):

    marker = f"STAGE 3 — STEP {step}:"
    matching_cells = []

    for cell in v1_notebooks_43["scheduling"]["cells"]:

        if cell.get("cell_type") != "code":
            continue

        streams = "\n".join(
            "".join(out.get("text", []))
            for out in cell.get("outputs", [])
            if out.get("output_type") == "stream"
        )

        if marker in streams:
            matching_cells.append(cell)

    assert len(matching_cells) == 1, (
        f"Expected one saved cell for {step}."
    )

    tables = []

    for output in matching_cells[0].get("outputs", []):

        data = output.get("data", {})

        if "text/plain" not in data:
            continue

        value = data["text/plain"]

        table_text = (
            "".join(value)
            if isinstance(value, list)
            else str(value)
        )

        if all(
            header in table_text
            for header in required_headers
        ):
            tables.append(table_text)

    assert len(tables) == 1, (
        f"Expected one table for {step}: "
        f"{required_headers}"
    )

    return tables[0]


def parsed_table_45(step, headers, pattern, columns):

    raw = saved_table_45(step, headers)

    records = []

    for line in raw.splitlines():
        match = re.fullmatch(
            pattern, line.strip()
        )

        if match:
            records.append(match.groups())

    assert records, (
        f"No table rows recovered for {step}."
    )

    return pd.DataFrame(
        records, columns=columns
    )

# ------------------------------------------------------------
# 2. PRELIMINARY FEASIBILITY — STAGE 3.1
# ------------------------------------------------------------

preliminary_pass_45 = number_45(
    "3.1.5",
    r"Passing necessary conditions:\s*(\d+)",
    "Initial preliminary pass"
)

preliminary_fail_45 = number_45(
    "3.1.5",
    r"Failing necessary conditions:\s*(\d+)",
    "Initial preliminary failures"
)

preliminary_recovered_45 = number_45(
    "3.1.6",
    r"Users with potential alternatives:\s*(\d+)",
    "Preliminary alternatives"
)

preliminary_deferred_45 = number_45(
    "3.1.7",
    r"Deferred users:\s*(\d+)",
    "Preliminary deferred"
)

preliminary_final_45 = number_45(
    "3.1.7",
    r"Unified preliminary-pass users:\s*(\d+)",
    "Preliminary final cohort"
)

v1_preliminary_checks_45 = parsed_table_45(
    "3.1.5",
    ["Condition", "Failed Users"],
    r"\d+\s+(\w+)\s+(\d+)\s+(\d+)",
    ["Condition", "Passed", "Failed"]
)

for col in ["Passed", "Failed"]:
    v1_preliminary_checks_45[col] = (
        v1_preliminary_checks_45[col].astype(int)
    )

v1_preliminary_duration_45 = parsed_table_45(
    "3.1.5",
    ["Failing_Necessary_Conditions"],
    r"\d+\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)",
    [
        "Duration", "Users",
        "Preliminary_pass", "Preliminary_fail"
    ]
).astype(int)

# ------------------------------------------------------------
# 3. WINDOW FEASIBILITY — STAGE 3.2
# ------------------------------------------------------------

initial_window_pass_45 = number_45(
    "3.2.1",
    r"Users with candidate windows:\s*(\d+)",
    "Initial window pass"
)

initial_window_fail_45 = number_45(
    "3.2.1",
    r"Users without candidate windows:\s*(\d+)",
    "Initial window failures"
)

joint_window_conflicts_45 = number_45(
    "3.2.2",
    r"no jointly feasible window:\s*(\d+)",
    "Joint-window conflicts"
)

three_channel_deferred_45 = number_45(
    "3.2.3",
    r"Three-channel cases deferred:\s*(\d+)",
    "Three-channel cases deferred"
)

window_alternative_users_45 = number_45(
    "3.2.3",
    r"Users included in alternative search:\s*(\d+)",
    "Users searched for alternatives"
)

window_recovered_45 = number_45(
    "3.2.4",
    r"Newly selected alternative users:\s*(\d+)",
    "Window alternative recovery"
)

window_final_45 = number_45(
    "3.2.4",
    r"Updated provisional window-pass cohort:\s*(\d+)",
    "Final window-pass cohort"
)

total_deferred_45 = number_45(
    "3.2.4",
    r"Unresolved users:\s*(\d+)",
    "Total deferred"
)

window_unresolved_45 = (
    initial_window_fail_45 - window_recovered_45
)

v1_window_diagnosis_45 = parsed_table_45(
    "3.2.1",
    ["Diagnosis", "Users"],
    r"\d+\s+(.+?)\s+(\d+)",
    ["Failure_diagnosis", "Users"]
)

v1_window_diagnosis_45["Users"] = (
    v1_window_diagnosis_45["Users"].astype(int)
)

v1_window_duration_45 = parsed_table_45(
    "3.2.1",
    ["Total_Passing_Windows"],
    r"\d+\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)",
    [
        "Duration", "Users_evaluated",
        "With_window", "Passing_windows",
        "Without_window"
    ]
).astype(int)

# ------------------------------------------------------------
# 4. COMPARE THE TWO INITIAL FAILURE STAGES
# ------------------------------------------------------------

v1_feasibility_by_duration_45 = (
    v1_preliminary_duration_45[
        ["Duration", "Preliminary_fail"]
    ]
    .merge(
        v1_window_duration_45[
            ["Duration", "Without_window"]
        ],
        on="Duration",
        validate="one_to_one"
    )
    .sort_values("Duration")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. NUMERICAL VALIDATION
# ------------------------------------------------------------

assert preliminary_pass_45 == 915
assert preliminary_fail_45 == 80
assert preliminary_recovered_45 == 10
assert preliminary_deferred_45 == 70
assert preliminary_final_45 == 925

assert (
    preliminary_pass_45 + preliminary_fail_45
    == 995
)

assert (
    preliminary_pass_45 + preliminary_recovered_45
    == preliminary_final_45
)

assert initial_window_pass_45 == 703
assert initial_window_fail_45 == 222
assert window_alternative_users_45 == 219
assert three_channel_deferred_45 == 3
assert window_recovered_45 == 46
assert window_final_45 == 749
assert window_unresolved_45 == 176
assert total_deferred_45 == 246
assert joint_window_conflicts_45 == 12

assert (
    initial_window_pass_45 + initial_window_fail_45
    == preliminary_final_45
)

assert (
    window_alternative_users_45
    + three_channel_deferred_45
    == initial_window_fail_45
)

assert (
    initial_window_pass_45 + window_recovered_45
    == window_final_45
)

assert (
    preliminary_deferred_45 + window_unresolved_45
    == total_deferred_45
)

assert v1_preliminary_duration_45["Users"].sum() == 995
assert (
    v1_preliminary_duration_45["Preliminary_fail"].sum()
    == 80
)

assert (
    v1_window_duration_45["Users_evaluated"].sum()
    == 925
)

assert (
    v1_window_duration_45["Without_window"].sum()
    == 222
)

assert v1_window_diagnosis_45["Users"].sum() == 222

# Match earlier Section 4.4 stage losses.
assert preliminary_deferred_45 == stage31_loss_44
assert window_unresolved_45 == stage32_loss_44

# ------------------------------------------------------------
# 6. CONCISE OUTPUT
# ------------------------------------------------------------

print("\nA. PRELIMINARY FEASIBILITY — STAGE 3.1")
print("-" * 72)

print(f"Original assignments   : 995")
print(f"Initial pass           : {preliminary_pass_45}")
print(f"Initial failure        : {preliminary_fail_45}")
print(f"Recovered alternatives : {preliminary_recovered_45}")
print(f"Remaining deferred     : {preliminary_deferred_45}")

print("\nOriginal failed conditions:")
print(
    v1_preliminary_checks_45[
        ["Condition", "Failed"]
    ].to_string(index=False)
)

print(
    "\nNote: Failed conditions can overlap "
    "for the same user."
)

print("\nB. WINDOW FEASIBILITY — STAGE 3.2")
print("-" * 72)

print(f"Users evaluated        : {preliminary_final_45}")
print(f"Initial window pass    : {initial_window_pass_45}")
print(f"Initial window failure : {initial_window_fail_45}")
print(f"Recovered alternatives : {window_recovered_45}")
print(f"Remaining unresolved   : {window_unresolved_45}")

print("\nInitial window failure diagnoses:")
print(
    v1_window_diagnosis_45.to_string(index=False)
)

print("\nAdditional observations:")
print(
    f"Joint-window conflicts : {joint_window_conflicts_45}"
)
print(
    f"3-channel cases deferred: {three_channel_deferred_45}"
)

print("\nC. INITIAL FAILURES BY DURATION")
print("-" * 72)

print(
    v1_feasibility_by_duration_45.to_string(index=False)
)

print(
    "\nNote: Preliminary and window failures refer "
    "to different successive cohorts."
)

print("\nD. FINAL PRE-SCHEDULING RECONCILIATION")
print("-" * 72)

print(f"Preliminary deferred  : {preliminary_deferred_45}")
print(f"Window unresolved     : {window_unresolved_45}")
print(f"Total deferred        : {total_deferred_45}")
print(f"Passed to scheduling  : {window_final_45}")

print("\nVALIDATION")
print("-" * 72)

print("Saved reports recovered : PASS")
print("Initial failure counts  : PASS")
print("Alternative recoveries  : PASS")
print("Duration reconciliation : PASS")
print("Final 246-user balance  : PASS")
print("Old notebooks executed  : NO")
print("Files written           : NONE")
print("Source modifications    : NONE")

print("\nSECTION 4.5 STATUS: PASS")


SECTION 4.5 — SCENARIO FEASIBILITY FAILURES

A. PRELIMINARY FEASIBILITY — STAGE 3.1
------------------------------------------------------------------------
Original assignments   : 995
Initial pass           : 915
Initial failure        : 80
Recovered alternatives : 10
Remaining deferred     : 70

Original failed conditions:
                 Condition  Failed
         duration_capacity       0
       active_day_capacity       6
     multichannel_capacity      24
    three_channel_capacity      62
    scenario_day_partition       0
scenario_channel_partition       0
       switch_count_bounds       0

Note: Failed conditions can overlap for the same user.

B. WINDOW FEASIBILITY — STAGE 3.2
------------------------------------------------------------------------
Users evaluated        : 925
Initial window pass    : 703
Initial window failure : 222
Recovered alternatives : 46
Remaining unresolved   : 176

Initial window failure diagnoses:
                           Failure_diagnosis  Use


## 4.6 Temporal & Channel Restrictions

**Research Question:** How did predefined channel-switching targets affect V1 scheduling feasibility after users passed the temporal-window checks?

**Objectives:**
- Recover exact switching feasibility results for the 749 window-qualified users.
- Identify failures caused by excessive, insufficient, or unreachable switching targets.
- Measure recovery through alternative windows and day allocations.
- Analyze switching failures and recoveries by attack duration.
- Reconcile the final exact scheduling search with the 670 simulated users.

**Inputs:**
- Saved Stage 3.3 execution reports from V1.
- Original scenario switching constraints.
- Validated population totals from Sections 4.4 and 4.5.

**Constraints:**
- Read-only analysis of saved reports.
- No execution of original scheduling algorithms.
- No recalculation of Notebook 02 rotation feasibility.
- No modifications to Baseline, Future, or V1 data.
- Do not interpret an initial switching failure as final infeasibility.

**Expected Output:** Validated switching failure categories, recovery rates, and final scheduling outcomes.


In [6]:

# ============================================================
# 4.6 — TEMPORAL & CHANNEL RESTRICTIONS
# Original switching failures and recovery analysis
# READ-ONLY
# ============================================================

import pandas as pd

print("=" * 72)
print("SECTION 4.6 — TEMPORAL & CHANNEL RESTRICTIONS")
print("=" * 72)

# ------------------------------------------------------------
# 1. READ ORIGINAL SWITCHING REPORTS
# ------------------------------------------------------------

initial_report_46 = saved_step_output_44(
    "STAGE 3 — STEP 3.3.3:"
)

recovery_report_46 = saved_step_output_44(
    "STAGE 3 — STEP 3.3.5:"
)

exact_report_46 = saved_step_output_44(
    "STAGE 3 — STEP 3.3.7:"
)

final_report_46 = saved_step_output_44(
    "STAGE 3 — STEP 3.3.9:"
)

def count_46(report, pattern, label):
    return report_number_44(report, pattern, label)

evaluated_46 = count_46(
    initial_report_46,
    r"Users evaluated:\s*(\d+)",
    "Users evaluated"
)

initial_pass_46 = count_46(
    initial_report_46,
    r"Exact switching target achievable:\s*(\d+)",
    "Initial switching pass"
)

initial_fail_46 = count_46(
    initial_report_46,
    r"Exact switching target not achievable:\s*(\d+)",
    "Initial switching failure"
)

recovered_46 = count_46(
    recovery_report_46,
    r"Recovered by alternative search:\s*(\d+)",
    "Alternative recovery"
)

unresolved_before_dp_46 = count_46(
    recovery_report_46,
    r"Still unresolved:\s*(\d+)",
    "Unresolved before exact search"
)

exact_recovered_46 = count_46(
    final_report_46,
    r"Additional exact DP schedules:\s*(\d+)",
    "Exact search recovery"
)

final_simulated_46 = count_46(
    final_report_46,
    r"Total consolidated users:\s*(\d+)",
    "Final simulated users"
)

# ------------------------------------------------------------
# 2. FIRST-PASS FEASIBILITY BY DURATION
# ------------------------------------------------------------

v1_switch_first_46 = parsed_table_45(
    "3.3.3",
    ["duration", "Users", "Passing", "Failing"],
    r"\d+\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)",
    ["Duration", "Evaluated", "Passed", "Failed"]
).astype(int)

# ------------------------------------------------------------
# 3. ORIGINAL SWITCHING FAILURE TYPES
# ------------------------------------------------------------

v1_switch_failures_46 = parsed_table_45(
    "3.3.4",
    ["Failure Type", "Users"],
    r"\d+\s+(.+?)\s+(\d+)",
    ["Failure_type", "Initial_failures"]
)

v1_switch_failures_46["Initial_failures"] = (
    v1_switch_failures_46["Initial_failures"].astype(int)
)

# ------------------------------------------------------------
# 4. RECOVERY BY FAILURE TYPE
# ------------------------------------------------------------

v1_switch_recovery_46 = parsed_table_45(
    "3.3.5",
    [
        "failure_type",
        "Initially_Failed",
        "Recovered",
        "Unresolved"
    ],
    r"\d+\s+(.+?)\s+(\d+)\s+(\d+)\s+(\d+)",
    [
        "Failure_type",
        "Initially_failed",
        "Recovered",
        "Unresolved"
    ]
)

for col in ["Initially_failed", "Recovered", "Unresolved"]:
    v1_switch_recovery_46[col] = (
        v1_switch_recovery_46[col].astype(int)
    )

# ------------------------------------------------------------
# 5. RECOVERY BY DURATION
# ------------------------------------------------------------

v1_switch_duration_recovery_46 = parsed_table_45(
    "3.3.5",
    [
        "duration",
        "Initially_Failed",
        "Recovered",
        "Unresolved"
    ],
    r"\d+\s+(\d+)\s+(\d+)\s+(\d+)\s+(\d+)",
    [
        "Duration",
        "Initially_failed",
        "Recovered",
        "Unresolved"
    ]
).astype(int)

v1_switch_duration_46 = v1_switch_first_46.merge(
    v1_switch_duration_recovery_46,
    on="Duration",
    validate="one_to_one"
)

v1_switch_duration_46["Initial_success_pct"] = (
    100
    * v1_switch_duration_46["Passed"]
    / v1_switch_duration_46["Evaluated"]
).round(1)

# ------------------------------------------------------------
# 6. FINAL EXACT SCHEDULING SEARCH
# ------------------------------------------------------------

v1_exact_search_46 = parsed_table_45(
    "3.3.7",
    ["Status", "Users"],
    r"\d+\s+(FOUND|INFEASIBLE|INCOMPLETE)\s+(\d+)",
    ["Status", "Users"]
)

v1_exact_search_46["Users"] = (
    v1_exact_search_46["Users"].astype(int)
)

exact_counts_46 = dict(
    zip(
        v1_exact_search_46["Status"],
        v1_exact_search_46["Users"]
    )
)

# ------------------------------------------------------------
# 7. VALIDATION
# ------------------------------------------------------------

assert evaluated_46 == 749
assert initial_pass_46 == 544
assert initial_fail_46 == 205
assert recovered_46 == 116
assert unresolved_before_dp_46 == 89
assert exact_recovered_46 == 10
assert final_simulated_46 == 670

assert initial_pass_46 + initial_fail_46 == evaluated_46

assert (
    v1_switch_failures_46["Initial_failures"].sum()
    == initial_fail_46
)

assert (
    v1_switch_recovery_46["Initially_failed"].sum()
    == initial_fail_46
)

assert (
    v1_switch_recovery_46["Recovered"].sum()
    == recovered_46
)

assert (
    v1_switch_recovery_46["Unresolved"].sum()
    == unresolved_before_dp_46
)

assert (
    v1_switch_recovery_46["Initially_failed"]
    == (
        v1_switch_recovery_46["Recovered"]
        + v1_switch_recovery_46["Unresolved"]
    )
).all()

assert (
    v1_switch_duration_46["Failed"]
    == v1_switch_duration_46["Initially_failed"]
).all()

assert v1_switch_duration_46["Evaluated"].sum() == 749
assert v1_switch_duration_46["Passed"].sum() == 544
assert v1_switch_duration_46["Recovered"].sum() == 116
assert v1_switch_duration_46["Unresolved"].sum() == 89

assert exact_counts_46 == {
    "FOUND": 10,
    "INFEASIBLE": 78,
    "INCOMPLETE": 1
}

assert sum(exact_counts_46.values()) == 89

assert (
    initial_pass_46
    + recovered_46
    + exact_recovered_46
    == final_simulated_46
)

assert (
    exact_counts_46["INFEASIBLE"]
    + exact_counts_46["INCOMPLETE"]
    == stage33_loss_44
)

assert (
    previously_deferred_44
    + evaluated_46
    == 995
)

# ------------------------------------------------------------
# 8. CONCISE OUTPUT
# ------------------------------------------------------------

print("\nA. INITIAL SWITCHING FEASIBILITY")
print("-" * 72)

print(f"Users evaluated        : {evaluated_46}")
print(f"Initial switching pass : {initial_pass_46}")
print(f"Initial switching fail : {initial_fail_46}")

print("\nB. FAILURE TYPES AND ALTERNATIVE RECOVERY")
print("-" * 72)

print(
    v1_switch_recovery_46.to_string(index=False)
)

print("\nC. SWITCHING RESULTS BY DURATION")
print("-" * 72)

print(
    v1_switch_duration_46[
        [
            "Duration",
            "Evaluated",
            "Passed",
            "Failed",
            "Recovered",
            "Unresolved",
            "Initial_success_pct"
        ]
    ].to_string(index=False)
)

print("\nD. EXACT SEARCH RESULTS")
print("-" * 72)

print(v1_exact_search_46.to_string(index=False))

print("\nE. FINAL SCHEDULING RECONCILIATION")
print("-" * 72)

print(f"Initial success           : {initial_pass_46}")
print(f"Alternative recovery      : {recovered_46}")
print(f"Exact search recovery     : {exact_recovered_46}")
print(f"Final simulated users     : {final_simulated_46}")
print(f"Final switching unresolved: {stage33_loss_44}")
print(f"Previously deferred       : {previously_deferred_44}")

print("\nVALIDATION")
print("-" * 72)

print("Switching population balance : PASS")
print("Failure categories           : PASS")
print("Alternative recovery         : PASS")
print("Duration-level totals        : PASS")
print("Exact scheduling outcomes    : PASS")
print("Final 995-user balance       : PASS")
print("Original source modified     : NO")
print("Files written                : NONE")

print("\nSECTION 4.6 STATUS: PASS")


SECTION 4.6 — TEMPORAL & CHANNEL RESTRICTIONS

A. INITIAL SWITCHING FEASIBILITY
------------------------------------------------------------------------
Users evaluated        : 749
Initial switching pass : 544
Initial switching fail : 205

B. FAILURE TYPES AND ALTERNATIVE RECOVERY
------------------------------------------------------------------------
                    Failure_type  Initially_failed  Recovered  Unresolved
  Insufficient possible switches                94         42          52
   Too many unavoidable switches                71         38          33
Unreachable target within bounds                40         36           4

C. SWITCHING RESULTS BY DURATION
------------------------------------------------------------------------
 Duration  Evaluated  Passed  Failed  Recovered  Unresolved  Initial_success_pct
        5        157     141      16          9           7                 89.8
       10        144     100      44         31          13                 69.


## 4.7 Rule Impact & Failure Attribution

**Research Question:** Which V1 feasibility stages contributed most to the final simulation shortfall, and how effective were the original recovery mechanisms?

**Objectives:**
- Quantify the final user losses at each feasibility stage.
- Measure alternative-recovery effectiveness.
- Summarize major preliminary, window, and switching failure patterns.
- Compare outcomes across predefined switching levels.
- Separate documented observations from causal interpretations.

**Evidence Limitations:**
- Stage-level losses are mutually exclusive.
- Preliminary condition failures may overlap.
- Window diagnoses and switching failures refer to different populations.
- Individual failure reasons must not be inferred from aggregate reports.
- Associations between rules and outcomes do not establish causation.

**Expected Output:** A consolidated, validated V1 failure-impact assessment to support V2 rule-design decisions.


In [7]:

# ============================================================
# 4.7 — RULE IMPACT & FAILURE ATTRIBUTION
# Consolidated evidence from Sections 4.4–4.6
# READ-ONLY
# ============================================================

import pandas as pd

print("=" * 72)
print("SECTION 4.7 — RULE IMPACT & FAILURE ATTRIBUTION")
print("=" * 72)

# ------------------------------------------------------------
# 1. STAGE-LEVEL FAILURE AND RECOVERY IMPACT
# ------------------------------------------------------------

v1_rule_impact_47 = pd.DataFrame([
    {
        "Stage": "3.1 Preliminary",
        "Evaluated": v1_assigned_count,
        "Initial_failed": preliminary_fail_45,
        "Recovered": preliminary_recovered_45,
        "Final_loss": preliminary_deferred_45,
        "Passed": preliminary_final_45
    },
    {
        "Stage": "3.2 Window",
        "Evaluated": preliminary_final_45,
        "Initial_failed": initial_window_fail_45,
        "Recovered": window_recovered_45,
        "Final_loss": window_unresolved_45,
        "Passed": window_final_45
    },
    {
        "Stage": "3.3 Switching",
        "Evaluated": evaluated_46,
        "Initial_failed": initial_fail_46,
        "Recovered": recovered_46 + exact_recovered_46,
        "Final_loss": stage33_loss_44,
        "Passed": final_simulated_46
    }
])

v1_rule_impact_47["Recovery_pct"] = (
    100
    * v1_rule_impact_47["Recovered"]
    / v1_rule_impact_47["Initial_failed"]
).round(1)

v1_rule_impact_47["Share_of_325_pct"] = (
    100
    * v1_rule_impact_47["Final_loss"]
    / v1_not_simulated
).round(1)

v1_rule_impact_47["Stage_success_pct"] = (
    100
    * v1_rule_impact_47["Passed"]
    / v1_rule_impact_47["Evaluated"]
).round(1)

# ------------------------------------------------------------
# 2. PRELIMINARY RULE FAILURE SIGNATURES
# ------------------------------------------------------------

v1_preliminary_impact_47 = (
    v1_preliminary_checks_45[
        ["Condition", "Failed"]
    ]
    .copy()
    .sort_values("Failed", ascending=False)
    .reset_index(drop=True)
)

# Condition failures may overlap across users.
assert (
    v1_preliminary_impact_47["Failed"].sum()
    >= preliminary_fail_45
)

# ------------------------------------------------------------
# 3. WINDOW FAILURE SIGNATURES
# ------------------------------------------------------------

v1_window_impact_47 = (
    v1_window_diagnosis_45
    .copy()
    .sort_values("Users", ascending=False)
    .reset_index(drop=True)
)

assert (
    v1_window_impact_47["Users"].sum()
    == initial_window_fail_45
)

# ------------------------------------------------------------
# 4. SWITCHING FAILURE SIGNATURES
# ------------------------------------------------------------

v1_switch_impact_47 = (
    v1_switch_failures_46
    .copy()
    .sort_values(
        "Initial_failures",
        ascending=False
    )
    .reset_index(drop=True)
)

assert (
    v1_switch_impact_47["Initial_failures"].sum()
    == initial_fail_46
)

# ------------------------------------------------------------
# 5. USER OUTCOMES BY PREDEFINED SWITCHING LEVEL
# ------------------------------------------------------------

v1_switch_level_47 = pd.crosstab(
    v1_user_outcomes_44["switching_level"],
    v1_user_outcomes_44["v1_result"]
).reindex(
    index=["Low", "Medium", "High"],
    columns=["Simulated", "Not simulated"],
    fill_value=0
)

v1_switch_level_47["Total"] = (
    v1_switch_level_47["Simulated"]
    + v1_switch_level_47["Not simulated"]
)

v1_switch_level_47["Success_pct"] = (
    100
    * v1_switch_level_47["Simulated"]
    / v1_switch_level_47["Total"]
).round(1)

assert v1_switch_level_47["Total"].sum() == 995
assert v1_switch_level_47["Simulated"].sum() == 670

# ------------------------------------------------------------
# 6. FINAL POPULATION RECONCILIATION
# ------------------------------------------------------------

total_initial_failures_47 = int(
    v1_rule_impact_47["Initial_failed"].sum()
)

total_recoveries_47 = int(
    v1_rule_impact_47["Recovered"].sum()
)

total_final_losses_47 = int(
    v1_rule_impact_47["Final_loss"].sum()
)

assert (
    total_initial_failures_47
    - total_recoveries_47
    == total_final_losses_47
)

assert total_final_losses_47 == 325
assert v1_assigned_count - total_final_losses_47 == 670

assert (
    v1_rule_impact_47["Evaluated"].tolist()
    == [995, 925, 749]
)

assert (
    v1_rule_impact_47["Passed"].tolist()
    == [925, 749, 670]
)

assert (
    v1_rule_impact_47["Final_loss"].tolist()
    == [70, 176, 79]
)

assert (
    exact_infeasible_44 + exact_incomplete_44
    == 79
)

# ------------------------------------------------------------
# 7. CONCISE REPORT
# ------------------------------------------------------------

print("\nA. FINAL STAGE IMPACT")
print("-" * 72)

print(
    v1_rule_impact_47[
        [
            "Stage",
            "Evaluated",
            "Initial_failed",
            "Recovered",
            "Final_loss",
            "Recovery_pct",
            "Share_of_325_pct"
        ]
    ].to_string(index=False)
)

print("\nB. PRELIMINARY FAILURE CONDITIONS")
print("-" * 72)

print(
    v1_preliminary_impact_47[
        v1_preliminary_impact_47["Failed"] > 0
    ].to_string(index=False)
)

print("\nNote: Preliminary conditions can overlap.")

print("\nC. INITIAL WINDOW FAILURE DIAGNOSES")
print("-" * 72)

print(v1_window_impact_47.to_string(index=False))

print("\nD. INITIAL SWITCHING FAILURE TYPES")
print("-" * 72)

print(v1_switch_impact_47.to_string(index=False))

print("\nE. OUTCOMES BY SWITCHING LEVEL")
print("-" * 72)

print(v1_switch_level_47.to_string())

print(
    "\nNote: Switching-level comparisons are "
    "descriptive and may be influenced by "
    "duration and channel requirements."
)

print("\nF. OVERALL FAILURE ACCOUNTING")
print("-" * 72)

print(f"Original assignments  : {v1_assigned_count}")
print(f"Initial stage failures: {total_initial_failures_47}")
print(f"Recovered across stages: {total_recoveries_47}")
print(f"Final non-simulated   : {total_final_losses_47}")
print(f"Final simulated       : {final_simulated_46}")

print("\nEVIDENCE INTERPRETATION")
print("-" * 72)

print(
    "Stage losses identify where users stopped, "
    "not a proven individual causal reason."
)
print(
    "Exact scheduling: 78 infeasible under V1 "
    "rules; 1 incomplete."
)
print(
    "V2 improvements must be validated against "
    "observed user/channel availability."
)

print("\nVALIDATION")
print("-" * 72)

print("Stage-level attribution : PASS")
print("Recovery reconciliation : PASS")
print("Rule signature totals   : PASS")
print("Switching-level totals  : PASS")
print("Final 995-user balance  : PASS")
print("Old notebooks executed : NO")
print("Files written          : NONE")
print("Source modifications   : NONE")

print("\nSECTION 4.7 STATUS: PASS")


SECTION 4.7 — RULE IMPACT & FAILURE ATTRIBUTION

A. FINAL STAGE IMPACT
------------------------------------------------------------------------
          Stage  Evaluated  Initial_failed  Recovered  Final_loss  Recovery_pct  Share_of_325_pct
3.1 Preliminary        995              80         10          70          12.5              21.5
     3.2 Window        925             222         46         176          20.7              54.2
  3.3 Switching        749             205        126          79          61.5              24.3

B. PRELIMINARY FAILURE CONDITIONS
------------------------------------------------------------------------
             Condition  Failed
three_channel_capacity      62
 multichannel_capacity      24
   active_day_capacity       6

Note: Preliminary conditions can overlap.

C. INITIAL WINDOW FAILURE DIAGNOSES
------------------------------------------------------------------------
                           Failure_diagnosis  Users
                         Ac


## 4.8 Final Validation & Documentation

**Objective:** Finalize the V1 Failure Analysis as an independent historical documentation notebook.

**Tasks:**
- Verify population and stage-level failure accounting.
- Confirm consistency between initial failures, recoveries, and final outcomes.
- Document the main limitations of V1 scenario design and scheduling.
- Summarize evidence-based lessons for future simulation design.
- Generate a standardized `notebook_summary.md` containing six sections.

**Documentation Policy:**
- No new analytical datasets.
- No changes to V1, Baseline, or Future data.
- No additional scenario feasibility computations.
- No dependencies from future notebooks on Notebook 04 outputs.

**Final Deliverable:** `reports/04_V1_Failure_Analysis/notebook_summary.md`


In [9]:

# ============================================================
# 4.8 — FINAL VALIDATION & DOCUMENTATION
# Notebook 04 — V1 Failure Analysis
# ============================================================

from pathlib import Path

print("=" * 72)
print("SECTION 4.8 — FINAL VALIDATION & DOCUMENTATION")
print("=" * 72)

# ------------------------------------------------------------
# 1. FINAL INTEGRITY VALIDATION
# ------------------------------------------------------------

checks = {
    "V1 assigned population":
        v1_assigned_count == 995,

    "V1 simulated population":
        v1_simulated_count == 670,

    "V1 non-simulated population":
        v1_not_simulated == 325,

    "Identical V1/V2 candidate identities":
        v1_assigned_users == current_candidate_users,

    "Preliminary stage loss":
        stage31_loss_44 == 70,

    "Window stage loss":
        stage32_loss_44 == 176,

    "Scheduling stage loss":
        stage33_loss_44 == 79,

    "Exclusive loss reconciliation":
        70 + 176 + 79 == 325,

    "Preliminary recovery":
        80 - 10 == 70,

    "Window recovery":
        222 - 46 == 176,

    "Switching recovery":
        205 - 116 - 10 == 79,

    "Final simulation reconciliation":
        995 - 325 == 670,

    "Exact scheduling outcomes":
        exact_infeasible_44 == 78
        and exact_incomplete_44 == 1,

    "Duration outcome reconciliation":
        v1_duration_outcomes_44["Simulated"].sum() == 670
        and
        v1_duration_outcomes_44["Not simulated"].sum() == 325,

    "Switching-level reconciliation":
        v1_switch_level_47["Total"].sum() == 995,

    "Failure-impact summary":
        int(v1_rule_impact_47["Final_loss"].sum()) == 325
}

failed_checks = [
    name for name, passed in checks.items()
    if not passed
]

assert not failed_checks, (
    f"Final validation failed: {failed_checks}"
)

# ------------------------------------------------------------
# 2. PREPARE DOCUMENTED RESULTS
# ------------------------------------------------------------

total_checks = len(checks)

preliminary_loss = stage31_loss_44
window_loss = stage32_loss_44
switching_loss = stage33_loss_44

preliminary_pct = 100 * preliminary_loss / 325
window_pct = 100 * window_loss / 325
switching_pct = 100 * switching_loss / 325

switching_recovered = recovered_46 + exact_recovered_46

window_active_day_failures = int(
    v1_window_diagnosis_45.loc[
        v1_window_diagnosis_45[
            "Failure_diagnosis"
        ].str.contains(
            "Active-day capacity",
            regex=False
        ),
        "Users"
    ].sum()
)

preliminary_three_channel_failures = int(
    v1_preliminary_checks_45.loc[
        v1_preliminary_checks_45["Condition"]
        == "three_channel_capacity",
        "Failed"
    ].iloc[0]
)

assert window_active_day_failures == 206
assert preliminary_three_channel_failures == 62
assert switching_recovered == 126

# ------------------------------------------------------------
# 3. GENERATE STANDARDIZED DOCUMENTATION
# ------------------------------------------------------------

summary = f"""# Notebook 04 — V1 Failure Analysis

## 1. Objective

Investigate why the previous MUBSIR attack simulation
reached only 670 simulated users out of 995 approved
scenario assignments.

This notebook is a standalone historical analysis.
Its outputs are not inputs to any subsequent notebook.

## 2. Data Scope

**Previous simulation:** Clear_Attack_Simulation (V1)

**Current reference:** New_Attack_Simulation

**Analyzed population:**
- V1 approved assignments: 995 users
- V1 simulated users: 670
- V1 non-simulated users: 325
- Current multi-channel candidates: 995

The V1 and current candidate populations have identical
user identities.

**Evidence sources:**
- V1 approved scenario catalog and assignments
- V1 scheduling notebook and saved execution reports
- V1 user-level Ground Truth
- Current candidate identity profile

All original sources were read without modification.

## 3. Key Findings

### 3.1 Overall Simulation Outcome

- Assigned users: 995
- Simulated users: 670
- Non-simulated users: 325
- Simulation completion rate: {670/995*100:.1f}%

### 3.2 Final Losses by Stage

| Stage | Final Loss | Share of 325 |
|---|---:|---:|
| Preliminary feasibility | {preliminary_loss} | {preliminary_pct:.1f}% |
| Window feasibility | {window_loss} | {window_pct:.1f}% |
| Exact switching/scheduling | {switching_loss} | {switching_pct:.1f}% |
| **Total** | **325** | **100.0%** |

Window feasibility accounted for the largest final
population loss.

### 3.3 Preliminary Feasibility

- Initial failures: 80
- Alternative recoveries: 10
- Final deferred users: 70
- Three-channel capacity failures: {preliminary_three_channel_failures}
- Multichannel capacity failures: 24
- Active-day capacity failures: 6

Preliminary failure conditions can overlap.

### 3.4 Window Feasibility

- Users evaluated: 925
- Initial window failures: 222
- Alternative recoveries: 46
- Final unresolved users: 176
- Initial window failures involving active-day capacity:
  {window_active_day_failures}

The strict activity requirements inside predefined
windows were a major source of initial failure.

### 3.5 Channel Switching

- Users evaluated: 749
- Initial switching failures: 205
- Alternative and exact-search recoveries:
  {switching_recovered}
- Final scheduling losses: 79

Initial switching failure types:
- Insufficient possible switches: 94
- Too many unavoidable switches: 71
- Unreachable target within bounds: 40

Final exact scheduling outcomes:
- Infeasible under V1 rules: 78
- Incomplete: 1

### 3.6 Scenario Design Findings

V1 predefined:
- Durations: 5, 10, 20, 30, 40, 50 workdays
- Switching targets based on fixed Low/Medium/High ranges
- Fixed active/off-day and same-day channel requirements

A total of 289 assigned users had long-duration
scenarios (40 or 50 days) requiring three-channel
activity on at least one common day.

Descriptively, simulation success was:
- Without same-day three-channel requirement: 71.4%
- With same-day three-channel requirement: 57.4%

This association does not establish independent
causality because scenario requirements and
durations were linked.

### 3.7 Lessons Learned

1. Derive feasible scenarios from observed user activity
   before assigning attack configurations.
2. Test durations against feasible user-specific windows.
3. Avoid unnecessary simultaneous multichannel requirements.
4. Derive achievable channel-switching patterns from
   observed activity instead of imposing fixed targets.
5. Preserve no-injection off-days without unnecessary
   predefined quotas.
6. Evaluate frequency/count-based stealth rules separately.
7. Preserve benign multi-channel users to reduce
   shortcut-learning risk.

These are design lessons, not proof that proposed
changes will succeed.

## 4. Validation

**Result:** PASS — {total_checks}/{total_checks} checks.

Validated:
- Population identities and counts
- Stage-level population losses
- Initial failures and recoveries
- Final scheduling reconciliation
- Duration and switching-level outcomes
- Consistency of the documented failure totals

No old injection notebook was executed.
No original source data was modified.

## 5. Final Deliverables

- `04_V1_Failure_Analysis.ipynb`
- `reports/04_V1_Failure_Analysis/notebook_summary.md`

No Parquet or CSV files are generated.

Notebook 04 remains independent and is not required
by subsequent notebooks.

## 6. Limitations & Next Step

**Limitations:**
- Aggregate stage losses do not prove the individual
  causal reason for every non-simulated user.
- Preliminary rule failures may overlap.
- A V1 infeasibility result applies to its original
  constraints, not necessarily to redesigned scenarios.
- The incomplete scheduling case is not a proven
  infeasible case.
- No V2 simulation improvement has been demonstrated
  by this documentation notebook.

**Next Step:**

Proceed independently to Notebook 05 —
Simulation Rules Feasibility.

Notebook 05 will use original data and the validated
outputs of Notebooks 00–03, not Notebook 04 artifacts.

The provisional V2 population target is 400 malicious
users selected from 995 multi-channel candidates,
subject to feasibility and subsequent ML evaluation.

Historical Baseline remains immutable, and simulation
focus remains on event frequency/counts only.
"""

# ------------------------------------------------------------
# 4. SAVE ONLY THE STANDARDIZED SUMMARY
# ------------------------------------------------------------

report_dir = (
    PROJECT_ROOT
    / "reports"
    / "04_V1_Failure_Analysis"
)

report_dir.mkdir(parents=True, exist_ok=True)

summary_path = report_dir / "notebook_summary.md"

summary_path.write_text(
    summary,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 5. VERIFY THE SAVED DOCUMENT
# ------------------------------------------------------------

saved_summary = summary_path.read_text(
    encoding="utf-8"
)

assert saved_summary == summary

required_sections = [
    "## 1. Objective",
    "## 2. Data Scope",
    "## 3. Key Findings",
    "## 4. Validation",
    "## 5. Final Deliverables",
    "## 6. Limitations & Next Step"
]

assert all(
    section in saved_summary
    for section in required_sections
)

assert len(required_sections) == 6

# ------------------------------------------------------------
# 6. FINAL CONCISE OUTPUT
# ------------------------------------------------------------

print("\nFINAL VALIDATION")
print("-" * 72)

print(f"Checks passed       : {total_checks}/{total_checks}")
print("Population balance  : PASS")
print("Failure accounting  : PASS")
print("Recovery accounting : PASS")
print("Documentation       : PASS")

print("\nFINAL DELIVERABLE")
print("-" * 72)

print(f"Summary file : {summary_path}")
print(f"Sections     : {len(required_sections)}/6")
print("Read-back    : PASS")

print("\nINTEGRITY")
print("-" * 72)

print("Original V1 files modified : NO")
print("Baseline/Future modified   : NO")
print("Analytical datasets saved  : NONE")
print("Future notebook dependency : NONE")

print("\nNOTEBOOK 04 STATUS: COMPLETE — PASS")


SECTION 4.8 — FINAL VALIDATION & DOCUMENTATION

FINAL VALIDATION
------------------------------------------------------------------------
Checks passed       : 16/16
Population balance  : PASS
Failure accounting  : PASS
Recovery accounting : PASS
Documentation       : PASS

FINAL DELIVERABLE
------------------------------------------------------------------------
Summary file : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\04_V1_Failure_Analysis\notebook_summary.md
Sections     : 6/6
Read-back    : PASS

INTEGRITY
------------------------------------------------------------------------
Original V1 files modified : NO
Baseline/Future modified   : NO
Analytical datasets saved  : NONE
Future notebook dependency : NONE

NOTEBOOK 04 STATUS: COMPLETE — PASS
